In [11]:
import pandas as pd #PDF TO TEXT PO2
import re
import pdfplumber

def extract_data_from_pdf(pdf_path):
    """
    Extract data from PDF file and return structured data
    """
    data = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Extract text from all pages
            full_text = ""
            for page in pdf.pages:
                full_text += page.extract_text() + "\n"
            
            print("Extracted text from PDF:")
            print("=" * 50)
            print(full_text)
            print("=" * 50)
            
            # Process the extracted text
            lines = full_text.split('\n')
            current_item = {}
            
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                    
                # Extract SrNo (looking for lines starting with numbers)
                sr_no_match = re.match(r'^\s*(\d+)\s*', line)
                if sr_no_match:
                    if current_item and 'SrNo' in current_item:  # Save previous item if exists
                        data.append(current_item)
                    current_item = {'SrNo': sr_no_match.group(1), 'OrderQty': 2}
                    
                # Extract Article code (patterns like 9-DD106-YG-25, 9-DD106-YG-7)
                article_match = re.search(r'([A-Z0-9\-]+YG\-?\d*\.?\d+)', line)
                if article_match and 'ArticleCode' not in current_item:
                    current_item['ArticleCode'] = article_match.group(1)
                    
                # Extract StyleCode from Your reference (text after RSBR2074-01, RSBR2074-03, etc.)
                style_match = re.search(r'RSBR2074\-(\d+)\s+([A-Z0-9]+)', line)
                if style_match:
                    current_item['StyleCode'] = style_match.group(2)
                    
                # Extract ItemSize from description (looking for numbers like 0.25, 0.07)
                # Multiple patterns to catch different formats
                size_patterns = [
                    r'pendant\s+(\d+\.\d+)',  # pendant 0.25
                    r'(\d+\.\d+)\s*ct',       # 0.25 ct
                    r'YG[-\s]*(\d+\.\d+)',    # YG-0.25 or YG 0.25
                ]
                
                for pattern in size_patterns:
                    size_match = re.search(pattern, line, re.IGNORECASE)
                    if size_match and 'ItemSize' not in current_item:
                        current_item['ItemSize'] = size_match.group(1)
                        break
            
            # Add the last item
            if current_item and 'SrNo' in current_item:
                data.append(current_item)
                
    except Exception as e:
        print(f"Error reading PDF file: {e}")
        return []
    
    return data

def create_excel_dataframe(data):
    """
    Create DataFrame with specified columns
    """
    if not data:
        print("No data extracted from PDF")
        return pd.DataFrame()
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Reorder columns as requested
    columns_order = ['SrNo', 'StyleCode', 'ItemSize', 'OrderQty']
    
    # Only include columns that exist in the data
    final_columns = [col for col in columns_order if col in df.columns]
    
    # Add missing columns with empty values
    for col in columns_order:
        if col not in df.columns:
            df[col] = ""
    
    return df[columns_order]

# Main execution
if __name__ == "__main__":
    # Specify your PDF file path here
   
    pdf_file_path = r"C:\Users\Pratik Mali\Desktop\tools\data\Uneek\Uneek_PO_3.pdf" # Change this to your actual PDF path
    
    # Extract data from PDF
    print(f"Reading PDF from: {pdf_file_path}")
    extracted_data = extract_data_from_pdf(pdf_file_path)
    
    if extracted_data:
        # Create DataFrame
        df = create_excel_dataframe(extracted_data)
        
        # Display the results
        print("\nExtracted Data:")
        print("=" * 50)
        print(df)
        
        # Save to Excel file
        # output_file = 'extracted_data.xlsx'
        # df.to_excel(output_file, index=False)
        # print(f"\nData successfully saved to '{output_file}'")
        
        # Display basic statistics
        print(f"\nExtraction Summary:")
        print(f"Total records extracted: {len(df)}")
        print(f"Columns: {list(df.columns)}")
        
    else:
        print("No data was extracted from the PDF file.")
    
    # Display the DataFrame in notebook
    # df

Reading PDF from: C:\Users\Pratik Mali\Desktop\tools\data\Uneek\Uneek_PO_3.pdf
Extracted text from PDF:
UNEEK JEWELRY, INC. PURCHASE ORDER
550 S HILL ST.
SUITE 1635
Los Angeles, ca 90013
Purchase Order Number: PO-108114
United States
Vendor 300580
Order Date 9/10/2025
Request Date 10/21/2025
To: Ship
Tushar Joshi MAIN WAREHOUSE UNEEK HQ
SHIMAYRA JEWELLERY 550 S. HILL STREET
SHIMAYRA JEWELLERY SUITE 1635
PLOT NO 62 SEEPZ LOS ANGELES, 90013
ANDHERI (E) CA
Mumbai, IN-400001
Confirm To Entered By Delivery Method Terms Buyer
Tushar Joshi MARVIN Net 45 days
Vendor Item Serial Customer
No Item No. Item # Picture No Description item # Stamp Size SO Pieces Weight
1 LVBE3000 R#18907 315793 5RD=.14CTW Uneek 6.5 1
4BAG=.33CTW 18KW
2 LVBAS2900Y R#16823 315794 24RD=0.08CTW 14KY Uneek 6.5 1
Total Lines: 2 Subtotal: 0.00
Line Discount 0.00
Comments:
Invoice Discount: 0.00
Tax: 0.00
Total: 0.00
“Vendor is fully responsible for any and all copyright infringements. Uneek Jewelry, Inc. places any potentia

In [ ]:
import pandas as pd
import re
import pdfplumber

def extract_uneek_data_with_columns(pdf_path):
    """
    Extract data from Uneek PDF and organize into 26 columns with user inputs
    """
    
    # Get user inputs
    print("=" * 60)
    print("UNEEK ORDER PROCESSING - USER INPUTS")
    print("=" * 60)
    
    style_code = input("Enter StyleCode: ").strip()
    order_qty = input("Enter OrderQty: ").strip()
    user_input1 = input("Enter first part for SpecialRemarks (e.g., COLVNM04Y): ").strip()
    user_input2 = input("Enter second part for SpecialRemarks (e.g., HI-VS): ").strip()
    stamp_instruction = input("Enter StampInstruction: ").strip()
    
    print("\n" + "=" * 60)
    print("Processing PDF...")
    print("=" * 60 + "\n")
    
    data = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Extract text from all pages
            full_text = ""
            for page in pdf.pages:
                full_text += page.extract_text() + "\n"
            
            print("Extracted text preview (first 1000 chars):")
            print("=" * 60)
            print(full_text[:1000])
            print("=" * 60 + "\n")
            
            # Extract PO Number
            po_match = re.search(r'Purchase Order Number:\s*(PO-\d+)', full_text)
            item_po_no = po_match.group(1) if po_match else ""
            print(f"PO Number found: {item_po_no}\n")
            
            # Process the extracted text line by line
            lines = full_text.split('\n')
            
            i = 0
            items_found = 0
            while i < len(lines):
                line = lines[i].strip()
                
                # More flexible pattern to match different SKU formats
                # Pattern 1: COLV format - 1 COLVBW1534 R061534 315447 26RD=0.50CTW...
                # Pattern 2: R format - 1 R1248OVY R#24503 315972 48RD=.27CTW...
                # Pattern 3: RB format - 3 RB1316Y BD10015 315974 7RD=1.91CTW...
                
                # Try to match: Number + SKU (various formats) + Item# + Serial# + Description
                item_match = re.match(r'^(\d+)\s+([A-Z]+[A-Z0-9\-]+)\s+([A-Z0-9#]+)\s+(\d+)\s+(.+)', line)
                
                if item_match:
                    sr_no = item_match.group(1)
                    sku_no = item_match.group(2)
                    item_no = item_match.group(3)
                    serial_no = item_match.group(4)
                    description_part1 = item_match.group(5).strip()
                    
                    items_found += 1
                    print(f"Item {items_found} found: SrNo={sr_no}, SKU={sku_no}, Desc={description_part1[:50]}...")
                    
                    # Look ahead to the next few lines for metal and size info
                    metal_raw = ""
                    size_from_text = ""
                    full_description = description_part1
                    
                    # Check next 3 lines for metal and size
                    for j in range(1, 4):
                        if i + j < len(lines):
                            next_line = lines[i + j].strip()
                            
                            # Stop if we hit another item or empty line
                            if re.match(r'^\d+\s+[A-Z]', next_line) or next_line == "":
                                break
                            
                            # Add to full description
                            full_description += " " + next_line
                            
                            # Extract metal (e.g., 18KW, 14KY, 18KY)
                            if not metal_raw:
                                metal_match = re.search(r'(\d+K[A-Z])', next_line)
                                if metal_match:
                                    metal_raw = metal_match.group(1)
                            
                            # Extract size (e.g., SZ6 or just 6.5)
                            if not size_from_text:
                                size_match = re.search(r'SZ(\d+)', next_line)
                                if size_match:
                                    size_from_text = size_match.group(1)
                                else:
                                    # Try to find decimal size like 6.5
                                    size_match2 = re.search(r'\b(\d+\.5)\b', next_line)
                                    if size_match2:
                                        size_from_text = size_match2.group(1)
                    
                    # Build CustomerProductionInstruction
                    # Extract the stone/diamond info from description_part1
                    stone_info_match = re.match(r'^([^\s]+(?:RD|=|CTW)[^\s]*(?:\s+[^\s]+(?:RD|=|CTW|TO|FIT)[^\s]*)*)', description_part1)
                    if stone_info_match:
                        stone_info = stone_info_match.group(1)
                        # Clean up
                        stone_info = re.sub(r'\s+[A-Z]+-\s*$', '', stone_info).strip()
                    else:
                        # Try to get first meaningful part
                        parts = description_part1.split()
                        stone_info = parts[0] if parts else description_part1
                    
                    # Construct CustomerProductionInstruction: stone_info + metal + size
                    if metal_raw and size_from_text:
                        customer_production_instruction = f"{stone_info} {metal_raw} SZ{size_from_text}"
                    elif metal_raw:
                        customer_production_instruction = f"{stone_info} {metal_raw}"
                    else:
                        customer_production_instruction = stone_info
                    
                    # Extract ItemSize
                    if size_from_text:
                        # Handle both integer and decimal sizes
                        if '.' in size_from_text:
                            item_size = f"US{size_from_text}"
                        else:
                            item_size = f"US{size_from_text.zfill(2)}"
                    else:
                        item_size = input(f"Enter ItemSize for item {sr_no} (SKU: {sku_no}): ").strip()
                    
                    # Process metal and tone
                    # Convert 18KW -> G18W, 14KY -> G14Y, etc.
                    if metal_raw:
                        karat_num = metal_raw[:-1]  # "18K"
                        tone = metal_raw[-1]  # "W"
                        metal = f"G{karat_num[:-1]}{tone}"  # "G" + "18" + "W" = "G18W"
                    else:
                        metal = ""
                        tone = ""
                    
                    # Determine metal_tone for SpecialRemarks
                    if metal:
                        karat = metal[1:-1]  # Extract karat number (e.g., "18" from "G18W")
                        if tone == 'W':
                            metal_tone = f"{karat}K WHITE GOLD"
                        elif tone == 'Y':
                            metal_tone = f"{karat}K YELLOW GOLD"
                        elif tone == 'R':
                            metal_tone = f"{karat}K ROSE GOLD"
                        else:
                            metal_tone = f"{karat}K GOLD"
                    else:
                        metal_tone = ""
                    
                    # Extract size for SpecialRemarks
                    if size_from_text:
                        size_display = f"SZ {size_from_text}"
                    else:
                        size_display = f"SZ {item_size}"
                    
                    # Build SpecialRemarks
                    special_remarks = f"{user_input1},{metal_tone},{size_display},DIA QLTY-{user_input2}"
                    
                    # Determine DesignProductionInstruction based on tone
                    if tone == 'W':
                        design_production_instruction = "White Rodium"
                    else:
                        design_production_instruction = "No rodium"
                    
                    # Create row with all 26 columns
                    row = {
                        'SrNo': sr_no,
                        'StyleCode': style_code,
                        'ItemSize': item_size,
                        'OrderQty': order_qty,
                        'OrderItemPcs': '1',
                        'Metal': metal,
                        'Tone': tone,
                        'ItemPoNo': item_po_no,
                        'ItemRefNo': '',
                        'StockType': '',
                        'MakeType': '',
                        'CustomerProductionInstruction': customer_production_instruction,
                        'SpecialRemarks': special_remarks,
                        'DesignProductionInstruction': design_production_instruction,
                        'StampInstruction': stamp_instruction,
                        'OrderGroup': '',
                        'Certificate': '',
                        'SKUNo': sku_no,
                        'Basestoneminwt': '',
                        'Basestonemaxwt': '',
                        'Basemetalminwt': '',
                        'Basemetalmaxwt': '',
                        'Productiondeliverydate': '',
                        'Expecteddeliverydate': '',
                        'SetPrice': '',
                        'StoneQuality': ''
                    }
                    
                    data.append(row)
                
                i += 1
            
            print(f"\nTotal items found: {items_found}")
    
    except Exception as e:
        print(f"Error reading PDF file: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()
    
    # Create DataFrame with all columns
    df = pd.DataFrame(data)
    
    return df

# Main execution
if __name__ == "__main__":
    # ============================================================
    # CHANGE THE PDF FILE PATH HERE
    # ============================================================
    pdf_file_path = r"C:\Users\Pratik Mali\Desktop\tools\data\Uneek\Uneek_Po_2.pdf"
    # ============================================================
    
    print(f"\n{'='*60}")
    print(f"Reading PDF from: {pdf_file_path}")
    print(f"{'='*60}\n")
    
    # Extract data with all columns
    df_complete = extract_uneek_data_with_columns(pdf_file_path)
    
    if not df_complete.empty:
        print("\n" + "=" * 60)
        print("EXTRACTION COMPLETE")
        print("=" * 60)
        print(f"\nTotal records extracted: {len(df_complete)}")
        print(f"Total columns: {len(df_complete.columns)}")
        print(f"\nColumns: {list(df_complete.columns)}")
        
        # Display first few rows
        print("\n" + "=" * 60)
        print("PREVIEW (First 5 rows)")
        print("=" * 60)
        print(df_complete.head())
        
        # Display key columns
        print("\n" + "=" * 60)
        print("Key Columns Preview")
        print("=" * 60)
        print(df_complete[['SrNo', 'SKUNo', 'Metal', 'Tone', 'ItemSize', 'CustomerProductionInstruction']].head(10))
        
        # Save to Excel with dynamic filename based on PO number
        po_num = df_complete['ItemPoNo'].iloc[0] if len(df_complete) > 0 else 'output'
        output_file = f'uneek_{po_num}_data.xlsx'
        df_complete.to_excel(output_file, index=False)
        print(f"\n✓ Data successfully saved to '{output_file}'")
        
    else:
        print("\n" + "=" * 60)
        print("WARNING: No data was extracted from the PDF file.")
        print("=" * 60)
        print("Please check:")
        print("1. The PDF file path is correct")
        print("2. The PDF contains item data in the expected format")
        print("3. Review the extracted text preview above")

# Display the complete DataFrame
df_complete


Reading PDF from: C:\Users\Pratik Mali\Desktop\tools\data\Uneek\Uneek_PO_3.pdf

UNEEK ORDER PROCESSING - USER INPUTS

Processing PDF...

Extracted text preview (first 500 chars):
UNEEK JEWELRY, INC. PURCHASE ORDER
550 S HILL ST.
SUITE 1635
Los Angeles, ca 90013
Purchase Order Number: PO-108114
United States
Vendor 300580
Order Date 9/10/2025
Request Date 10/21/2025
To: Ship
Tushar Joshi MAIN WAREHOUSE UNEEK HQ
SHIMAYRA JEWELLERY 550 S. HILL STREET
SHIMAYRA JEWELLERY SUITE 1635
PLOT NO 62 SEEPZ LOS ANGELES, 90013
ANDHERI (E) CA
Mumbai, IN-400001
Confirm To Entered By Delivery Method Terms Buyer
Tushar Joshi MARVIN Net 45 days
Vendor Item Serial Customer
No Item No. Item #

PO Number found: PO-108114

No data was extracted from the PDF file.


""
